# Week 6 -- Function 5: GP + UCB Exploitation (4D)
**Chemical Process Yield** — unimodal, maximise Y. W4 peak at 3531, push further.

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 20
DATA_PATH_X  = '../data/function_5/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_5/initial_outputs.npy'

# Trend: x1 DOWN, x2/x3/x4 UP toward 1.0 = higher yield
# W5 SVM/NN pulled query back from W4 peak -- dropped from 3531 to 2081
weekly_log = [
    ([0.209005, 0.838746, 0.859156, 0.882439], 984.4093632633864),   # W1: x1=0.209
    ([0.204881, 0.877830, 0.879582, 0.870578], 1192.2995655092311),  # W2: x1=0.205
    ([0.204881, 0.877830, 0.879582, 0.870578], 1192.2995655092311),  # W3: plateau
    ([0.133474, 0.977830, 0.979582, 0.970578], 3531.0551965197214),  # W4: BEST x1=0.133
    ([0.154959, 0.906482, 0.957872, 0.906714], 2081.339032339055),   # W5: x1 rose, dropped
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Already up-to-date.
X: (25, 4) | Y: (25,)
Week 6 | 25 total observations (20 initial + 5 added)


In [2]:
# Cell 3: Load data and inspect -- Function 5: Chemical Process Yield (4D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by Y descending -- higher Y = better yield
pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 80)
print('  All observations (sorted descending by Y)   x1 column shows key pattern')
print('=' * 80)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.6f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:>10.2f}   x1={x_val[0]:.3f}{marker}')
print('=' * 80)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.8f}' for v in best_X)
print(f'\n  Best Y*  = {best_Y:.4f}')
print(f'  Best X*  = [{best_x_str}]')
print(f'  x1*      = {best_X[0]:.6f}  (lower = better)')

Input  shape : (25, 4)
Output shape : (25,)

  All observations (sorted descending by Y)   x1 column shows key pattern
  [ 1]  X = [0.133474, 0.977830, 0.979582, 0.970578]   Y =    3531.06   x1=0.133  <-- best
  [ 2]  X = [0.154959, 0.906482, 0.957872, 0.906714]   Y =    2081.34   x1=0.155
  [ 3]  X = [0.204881, 0.877830, 0.879582, 0.870578]   Y =    1192.30   x1=0.205
  [ 4]  X = [0.204881, 0.877830, 0.879582, 0.870578]   Y =    1192.30   x1=0.205
  [ 5]  X = [0.224189, 0.846480, 0.879484, 0.878516]   Y =    1088.86   x1=0.224
  [ 6]  X = [0.209005, 0.838746, 0.859156, 0.882439]   Y =     984.41   x1=0.209
  [ 7]  X = [0.119879, 0.862540, 0.643331, 0.849804]   Y =     431.61   x1=0.120
  [ 8]  X = [0.438933, 0.774092, 0.378167, 0.933696]   Y =     355.81   x1=0.439
  [ 9]  X = [0.836478, 0.193610, 0.663893, 0.785649]   Y =     258.37   x1=0.836
  [10]  X = [0.463442, 0.630025, 0.107906, 0.957644]   Y =     233.22   x1=0.463
  [11]  X = [0.352356, 0.322242, 0.116979, 0.473113]   Y =   

In [3]:
# Cell 4: Fit GP
# F5 values range ~100 to 3531 -- fit on raw Y directly.
# length_scale=0.3 so GP can extrapolate from training data into the near-boundary grid.
# alpha=1e-4 for slight noise tolerance.
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

kernel = RBF(length_scale=0.3, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-4)
gp.fit(X, Y)

# W4 anchor
w4_anchor = np.array([0.133474, 0.977830, 0.979582, 0.970578])

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Alpha (noise)           : 1e-4')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
print('  Sanity check at best known X*:')
best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'    X*                     = [{best_x_str}]')
print(f'    GP predicted mean      = {pred_mean[0]:.2f}')
print(f'    Actual Y*              = {best_Y:.2f}')
print(f'    GP predicted std       = {pred_std[0]:.6f}')
pred_w4, std_w4 = gp.predict(w4_anchor.reshape(1, -1), return_std=True)
w4_str = ', '.join(f'{v:.3f}' for v in w4_anchor)
print(f'  GP at W4 anchor [{w4_str}]:')
print(f'    predicted mean         = {pred_w4[0]:.2f}  (actual: 3531.06)')
print(f'    predicted std          = {std_w4[0]:.6f}')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.3)
  Alpha (noise)           : 1e-4
  Log-marginal-likelihood : -16168894.0244
  Sanity check at best known X*:
    X*                     = [0.133474, 0.977830, 0.979582, 0.970578]
    GP predicted mean      = 3529.21
    Actual Y*              = 3531.06
    GP predicted std       = 0.009991
  GP at W4 anchor [0.133, 0.978, 0.980, 0.971]:
    predicted mean         = 3529.21  (actual: 3531.06)
    predicted std          = 0.009991


In [4]:
# Cell 5: GP UCB Acquisition -- push toward low x1, high x2/x3/x4
# W4 at x1=0.133, x2/x3/x4~0.97 gave 3531. Push further in that direction.

x1 = np.linspace(0.05, 0.20, 20)
x2 = np.linspace(0.92, 1.00, 20)
x3 = np.linspace(0.92, 1.00, 20)
x4 = np.linspace(0.92, 1.00, 20)
xx1, xx2, xx3, xx4 = np.meshgrid(x1, x2, x3, x4)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel(), xx3.ravel(), xx4.ravel()])

gp_mean, gp_std = gp.predict(X_grid, return_std=True)

# beta=1.0: exploit the known peak direction
beta = 1.0
ucb_scores = gp_mean + beta * gp_std

best_idx = np.argmax(ucb_scores)
next_x = X_grid[best_idx]
portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print('=' * 72)
print('  GP UCB Acquisition (beta=1.0, exploitation)')
print('=' * 72)
print(f'  Grid : x1=[0.05,0.20] x x2=[0.92,1.00] x x3=[0.92,1.00] x x4=[0.92,1.00]')
print(f'         20^4 = 160,000 points')
print(f'  Beta : {beta}  (exploitation)')
next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Next query   : [{next_str}]')
print(f'  x1 proposed  : {next_x[0]:.6f}  (target: < 0.20)')
print(f'  UCB score    : {ucb_scores[best_idx]:.2f}')
print(f'  GP mean      : {gp_mean[best_idx]:.2f}')
print(f'  GP std       : {gp_std[best_idx]:.4f}')
print('=' * 72)

  GP UCB Acquisition (beta=1.0, exploitation)
  Grid : x1=[0.05,0.20] x x2=[0.92,1.00] x x3=[0.92,1.00] x x4=[0.92,1.00]
         20^4 = 160,000 points
  Beta : 1.0  (exploitation)
  Next query   : [0.152632, 1.000000, 1.000000, 1.000000]
  x1 proposed  : 0.152632  (target: < 0.20)
  UCB score    : 4054.26
  GP mean      : 4054.18
  GP std       : 0.0745


In [5]:
# Cell 6: Sanity check -- stay near W4 breakthrough direction

w4_anchor = np.array([0.133474, 0.977830, 0.979582, 0.970578])
dist_to_w4   = np.linalg.norm(next_x - w4_anchor)
dist_to_best = np.linalg.norm(next_x - best_X)

in_zone = (
    0.05 <= next_x[0] <= 0.20 and
    0.92 <= next_x[1] <= 1.00 and
    0.92 <= next_x[2] <= 1.00 and
    0.92 <= next_x[3] <= 1.00
)
x1_low       = next_x[0] < 0.25
x234_high    = all(next_x[i] > 0.85 for i in [1, 2, 3])

print('=' * 72)
print('  Sanity Check')
print('=' * 72)
next_str = ', '.join(f'{v:.6f}' for v in next_x)
w4_str   = ', '.join(f'{v:.6f}' for v in w4_anchor)
best_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'  Proposed query  : [{next_str}]')
print(f'  W4 anchor       : [{w4_str}]  (Y=3531, best ever)')
print(f'  Best known X*   : [{best_str}]  (Y={best_Y:.2f})')
print(f'  Distance to W4  : {dist_to_w4:.6f}')
print(f'  Distance to X*  : {dist_to_best:.6f}')
print(f'  x1 < 0.25       : {x1_low}  (x1={next_x[0]:.6f})')
print(f'  x2/x3/x4 > 0.85 : {x234_high}')
print(f'  Inside zone     : {in_zone}')
print()

if dist_to_w4 > 0.40:
    print('  WARNING: query is far from W4 breakthrough (> 0.40). Risk of losing peak!')
else:
    print('  OK: query is close to W4 breakthrough.')

if not x1_low:
    print('  WARNING: x1 >= 0.25 -- trend says lower x1 = higher yield!')
else:
    print('  OK: x1 is low, consistent with improving trend.')

if not x234_high:
    print('  WARNING: x2/x3/x4 not all > 0.85 -- trend says push these toward 1.0!')
else:
    print('  OK: x2/x3/x4 are high, consistent with improving trend.')

if not in_zone:
    print('  WARNING: query is outside target zone bounds!')
else:
    print('  OK: query is inside target zone.')
print('=' * 72)

  Sanity Check
  Proposed query  : [0.152632, 1.000000, 1.000000, 1.000000]
  W4 anchor       : [0.133474, 0.977830, 0.979582, 0.970578]  (Y=3531, best ever)
  Best known X*   : [0.133474, 0.977830, 0.979582, 0.970578]  (Y=3531.06)
  Distance to W4  : 0.046272
  Distance to X*  : 0.046272
  x1 < 0.25       : True  (x1=0.152632)
  x2/x3/x4 > 0.85 : True
  Inside zone     : True

  OK: query is close to W4 breakthrough.
  OK: x1 is low, consistent with improving trend.
  OK: x2/x3/x4 are high, consistent with improving trend.
  OK: query is inside target zone.


In [6]:
print('=' * 72)
print('  SUMMARY -- Week 6 Bayesian Optimisation')
print('=' * 72)
print('  Function        : 5 -- Chemical Process Yield (4D)')
print('  Objective       : Maximise Y (higher = better yield)')
print(f'  Method          : GP UCB (beta=1.0, 4D grid, raw Y fit, length_scale=0.3)')
print(f'  Key trend       : x1 lower + x2/x3/x4 higher toward 1.0 = higher yield')
print(f'  W4 peak         : 3531 at [0.133, 0.978, 0.980, 0.971] -- pushing further')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X*         : [{best_x_s}]')
print(f'  Best Y*         : {best_Y:.4f}')
print()
print(f'  Next query      : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 6 Bayesian Optimisation
  Function        : 5 -- Chemical Process Yield (4D)
  Objective       : Maximise Y (higher = better yield)
  Method          : GP UCB (beta=1.0, 4D grid, raw Y fit, length_scale=0.3)
  Key trend       : x1 lower + x2/x3/x4 higher toward 1.0 = higher yield
  W4 peak         : 3531 at [0.133, 0.978, 0.980, 0.971] -- pushing further

  Best X*         : [0.133474, 0.977830, 0.979582, 0.970578]
  Best Y*         : 3531.0552

  Next query      : [0.152632, 1.000000, 1.000000, 1.000000]

  Portal submission string:
  >>> 0.152632-1.000000-1.000000-1.000000 <<<


## Week 6 -- Reflection

**Function**: F5 -- Chemical Process Yield (4D)

### Strategy change from Week 5
- Removed NN/PyTorch: tight trust radius (0.05) caused W5 to retreat from W4's peak.
- Removed SVM: was endorsing conservative, centre-biased points.
- Raw Y fit, length_scale=0.3: GP extrapolates correctly into the near-boundary grid.
- Grid pushes beyond W4: x1=[0.05,0.20], x2/x3/x4=[0.92,1.00].

### Trend (weekly queries)
| Week | x1 | x2/x3/x4 avg | Y |
|------|-----|-------------|----|
| W1 | 0.209 | 0.860 | 984 |
| W2 | 0.205 | 0.876 | 1192 |
| W4 | 0.133 | 0.976 | **3531** |
| W5 | 0.155 | 0.924 | 2081 |

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 7
*(fill in after reflecting on result)*